# SageMaker Processing Job — BYOC con scikit-learn

Preprocesamiento del dataset **1C Company — Predict Future Sales** usando un container
propio (BYOC) con scikit-learn.

### Flujo de datos

```
S3 (sales_train.csv, items.csv)  →  /opt/ml/processing/input/  →  preprocess.py  →  /opt/ml/processing/output/  →  S3 (features listos)
```

El script no lee ni escribe S3 directamente — solo trabaja con rutas locales.
SageMaker gestiona la transferencia automáticamente.

### Contenidos
1. [Setup](#setup)
2. [Carga del dataset a S3](#dataset)
3. [Construcción del container](#container)
4. [Script de preprocessing](#script)
5. [Ejecutar el processing job](#run)
6. [Inspeccionar el output](#inspect)

## 1. Setup {#setup}

Sesión de SageMaker, IAM role, bucket y prefijos S3.

In [ ]:
from time import gmtime, strftime
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

prefix = "sagemaker/1c-processing"
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

input_prefix  = prefix + "/input/raw"
output_prefix = prefix + "/input/preprocessed"

print(f"Role    : {role}")
print(f"Bucket  : {bucket}")
print(f"Region  : {region}")
print(f"Account : {account_id}")
print(f"Prefix  : {prefix}")

## 2. Carga del dataset a S3 {#dataset}

Sube `sales_train.csv` e `items.csv` al bucket de SageMaker como datos crudos.

In [ ]:
# Ajusta estas rutas si tus CSVs están en otro directorio
for filename in ["sales_train.csv", "items.csv"]:
    uri = sagemaker_session.upload_data(
        path=filename,
        bucket=bucket,
        key_prefix=input_prefix
    )
    print(f"Subido: {uri}")

## 3. Construcción del container {#container}

El container BYOC es una imagen `python:3.12-slim` con scikit-learn, pandas y numpy.
No requiere bootstrapping — SageMaker inyecta y ejecuta el script directamente con `python3`.

In [ ]:
%cd src/preprocessing
!docker build --network sagemaker -t 1c-preprocessing .
%cd ../..

### Push a Amazon ECR

In [ ]:
ecr_repository = "1c-preprocessing"
tag = ":latest"
uri_suffix = "amazonaws.com"

repository_uri = "{}.dkr.ecr.{}.{}/{}".format(
    account_id, region, uri_suffix, ecr_repository + tag
)

print(f"Image URI: {repository_uri}")

In [ ]:
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!aws ecr create-repository --repository-name {ecr_repository}
!docker tag {ecr_repository + tag} {repository_uri}
!docker push {repository_uri}

## 4. Script de preprocessing {#script}

La celda siguiente escribe `preprocess.py` al disco. SageMaker lo sube a S3 y lo inyecta
en el container en tiempo de ejecución — no necesita estar baked en la imagen Docker.

**¿Qué hace el script?**

- **Carga:** lee `sales_train.csv` e `items.csv` desde `/opt/ml/processing/input/`.
- **Limpieza:** elimina registros con precio no positivo o unidades negativas.
- **Agregación mensual:** agrupa ventas diarias a nivel `date_block_num / shop_id / item_id` y clipea a 20.
- **Enriquecimiento:** agrega `item_category_id` desde `items.csv`.
- **Grid completo:** construye todas las combinaciones activas de shop/item/mes, rellenando con 0 donde no hubo ventas.
- **Lag features:** genera lags 1, 3, 6 y 12 meses del target.
- **Casos completos:** elimina filas con NaN en los lags (primeros meses sin historia).
- **Split train/test:** divide en 70% train / 30% test.
- **Output:** guarda 4 archivos CSV en `/opt/ml/processing/output/` que SageMaker copia a S3.

In [ ]:
%%writefile preprocess.py
from __future__ import print_function, unicode_literals
import argparse
import os

import pandas as pd
from sklearn.model_selection import train_test_split

# ── Constantes ────────────────────────────────────────────────────────────────
COLUMN_UNITS       = "item_cnt_day"
COLUMN_UNITS_MONTH = "item_cnt_month"
COLUMN_PRICE       = "item_price"
CLIP               = 20
LAGS               = [1, 3, 6, 12]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-test-split-ratio", type=float, default=0.3)
    args, _ = parser.parse_known_args()

    # ── Rutas SageMaker ───────────────────────────────────────────────────────
    input_dir  = "/opt/ml/processing/input"
    train_dir  = "/opt/ml/processing/output/train"
    test_dir   = "/opt/ml/processing/output/test"

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir,  exist_ok=True)

    # ── 1. Carga ──────────────────────────────────────────────────────────────
    print("Loading sales data...")
    sales = pd.read_csv(os.path.join(input_dir, "sales_train.csv"))
    print("Sales loaded: {} records".format(len(sales)))

    print("Loading items data...")
    items = pd.read_csv(os.path.join(input_dir, "items.csv"))
    print("Items loaded: {} records".format(len(items)))

    # ── 2. Limpieza ───────────────────────────────────────────────────────────
    print("Cleaning sales data...")
    sales = sales[(sales[COLUMN_PRICE] > 0) & (sales[COLUMN_UNITS] >= 0)].copy()
    print("After cleaning: {} records".format(len(sales)))

    # ── 3. Agregación mensual ─────────────────────────────────────────────────
    print("Aggregating to monthly level...")
    monthly = (
        sales.groupby(["date_block_num", "shop_id", "item_id"], as_index=False)
        .agg({COLUMN_UNITS: "sum"})
        .rename(columns={COLUMN_UNITS: COLUMN_UNITS_MONTH})
    )
    monthly[COLUMN_UNITS_MONTH] = monthly[COLUMN_UNITS_MONTH].clip(0, CLIP)
    print("Monthly aggregation: {} records".format(len(monthly)))

    # ── 4. Agrega item_category_id ────────────────────────────────────────────
    print("Adding item category information...")
    monthly = monthly.merge(
        items[["item_id", "item_category_id"]], on="item_id", how="left"
    )

    # ── 5. Grid completo ──────────────────────────────────────────────────────
    print("Building complete shop-item-month grid...")
    monthly_sorted = monthly.sort_values(["shop_id", "item_id", "date_block_num"])
    total_months = sorted(monthly_sorted["date_block_num"].unique())
    comb_active  = monthly_sorted[["shop_id", "item_id"]].drop_duplicates()

    grid = pd.concat(
        [comb_active.assign(date_block_num=m) for m in total_months],
        ignore_index=True
    )[["date_block_num", "shop_id", "item_id"]]

    grid = grid.merge(
        monthly_sorted[["date_block_num", "shop_id", "item_id",
                         COLUMN_UNITS_MONTH, "item_category_id"]],
        on=["date_block_num", "shop_id", "item_id"],
        how="left",
    )
    grid[COLUMN_UNITS_MONTH]   = grid[COLUMN_UNITS_MONTH].fillna(0)
    grid["item_category_id"]   = grid["item_category_id"].fillna(
        grid["item_category_id"].mode()[0]
    )
    grid = grid.sort_values(["shop_id", "item_id", "date_block_num"])
    print("Grid built: {} records".format(len(grid)))

    # ── 6. Lag features ───────────────────────────────────────────────────────
    print("Generating lag features: {}...".format(LAGS))
    for lag in LAGS:
        grid["lag_{}".format(lag)] = (
            grid.groupby(["shop_id", "item_id"])[COLUMN_UNITS_MONTH].shift(lag)
        )

    # ── 7. Filtra casos completos ─────────────────────────────────────────────
    lag_cols = ["lag_{}".format(l) for l in LAGS]
    grid = grid.dropna(subset=lag_cols)
    print("After filtering incomplete cases: {} records".format(len(grid)))

    # ── 8. Split train / test ─────────────────────────────────────────────────
    feature_cols = ["date_block_num", "shop_id", "item_id", "item_category_id"] + lag_cols
    X = grid[feature_cols]
    y = grid[COLUMN_UNITS_MONTH]

    split_ratio = args.train_test_split_ratio
    print("Splitting data with ratio {}".format(split_ratio))
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=split_ratio, random_state=0
    )

    print("Train data shape: {}".format(X_train.shape))
    print("Test data shape : {}".format(X_test.shape))

    # ── 9. Guarda los 4 archivos de output ────────────────────────────────────
    train_features_path = os.path.join(train_dir, "train_features.csv")
    train_labels_path   = os.path.join(train_dir, "train_labels.csv")
    test_features_path  = os.path.join(test_dir,  "test_features.csv")
    test_labels_path    = os.path.join(test_dir,  "test_labels.csv")

    print("Saving train features to {}".format(train_features_path))
    X_train.to_csv(train_features_path, header=False, index=False)

    print("Saving test features to {}".format(test_features_path))
    X_test.to_csv(test_features_path, header=False, index=False)

    print("Saving train labels to {}".format(train_labels_path))
    y_train.to_csv(train_labels_path, header=False, index=False)

    print("Saving test labels to {}".format(test_labels_path))
    y_test.to_csv(test_labels_path, header=False, index=False)

    print("Preprocessing complete.")

## 5. Ejecutar el Processing Job {#run}

`ScriptProcessor` ejecuta el script en el container BYOC con una sola instancia.

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `instance_count` | `1` | Una sola instancia — sin topología scheduler/worker |
| `instance_type` | `ml.m5.large` | 2 vCPUs, 8 GB RAM |
| `command` | `["python3"]` | SageMaker ejecuta `python3 preprocess.py [args]` |
| `ProcessingInput` | `s3://.../raw` → `/opt/ml/processing/input/` | SageMaker descarga los CSVs antes de iniciar |
| `ProcessingOutput (train)` | `/opt/ml/processing/output/train` → `s3://.../train` | Features y labels de entrenamiento |
| `ProcessingOutput (test)` | `/opt/ml/processing/output/test` → `s3://.../test` | Features y labels de evaluación |

In [ ]:
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

processor = ScriptProcessor(
    base_job_name="1c-preprocessing",
    image_uri=repository_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1200,
)

processor.run(
    code="preprocess.py",
    arguments=["--train-test-split-ratio", "0.3"],
    inputs=[
        ProcessingInput(
            source="s3://{}/{}".format(bucket, input_prefix),
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination="s3://{}/{}/train".format(bucket, output_prefix),
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination="s3://{}/{}/test".format(bucket, output_prefix),
        ),
    ],
    logs=True,
)

## 6. Inspeccionar el output {#inspect}

Revisa las primeras filas del dataset transformado para verificar que el preprocessing fue exitoso.

In [ ]:
# Vista rápida con AWS CLI
print("Top 5 rows from s3://{}/{}/train/".format(bucket, output_prefix))
!aws s3 cp --quiet s3://{bucket}/{output_prefix}/train/train_features.csv - | head -n5

In [ ]:
import pandas as pd

s3_train_features = "s3://{}/{}/train/train_features.csv".format(bucket, output_prefix)
df_train = pd.read_csv(s3_train_features, header=None)
print("Train features shape:", df_train.shape)
df_train.head()

In [ ]:
s3_test_features = "s3://{}/{}/test/test_features.csv".format(bucket, output_prefix)
df_test = pd.read_csv(s3_test_features, header=None)
print("Test features shape:", df_test.shape)
df_test.head()

Los archivos de salida pueden usarse directamente como input para un training job.